# BUỔI 2 - MIN EDIT DISTANCE, N-GRAMS VÀ SPELLING CORRECTION

## Cài đặt thư viện nltk

In [1]:
%pip install nltk pandas
import nltk
from nltk.metrics import distance

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
nltk.download("brown")
nltk.download("words")

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [3]:
import math
import random
import pandas as pd
from nltk.corpus import brown, words

# 1. MÔ HÌNH NGÔN NGỮ 1: EDIT DISTANCE

- Muốn chuyển chuỗi này sang chuỗi khác thì dùng 1 trong 3 phép toán: **Insertion**, **Deletion**, **Substitution**
- Chi phí mỗi phép toán là 1; riêng Substitution theo Levenshtein có cost là 2
- Yếu tố tìm đường đi tối thiểu: Initial state, Operators, Goal state, Path cost
- Dynamic programming: lập bảng *The Edit Distance Table*, khởi tạo D(i,0) = i và D(0,j) = j

## 1. Mã hóa và thuật toán Minimum Edit Distance cho cấp từ vựng

In [4]:
source = "intention"
target = "execution"

In [5]:
distance.edit_distance(source, target, substitution_cost = 1)

5

In [6]:
distance.edit_distance(source, target, substitution_cost = 2)

8

In [7]:
distance.edit_distance_align(source, target, substitution_cost = 2)

[(0, 0),
 (1, 0),
 (2, 1),
 (3, 2),
 (4, 3),
 (4, 4),
 (5, 5),
 (6, 6),
 (7, 7),
 (8, 8),
 (9, 9)]

### Cài đặt lại thuật toán theo slide 14 (Levenshtein)

In [8]:
def med(source, target, sub_cost = 2):
    D = [[0] * (len(target) + 1) for _ in range(len(source) + 1)]
    for i in range(len(source) + 1):
        D[i][0] = i
    for j in range(len(target) + 1):
        D[0][j] = j
    for i in range(1, len(source) + 1):
        for j in range(1, len(target) + 1):
            cost = 0 if source[i - 1] == target[j - 1] else sub_cost
            D[i][j] = min(D[i - 1][j] + 1,
                          D[i][j - 1] + 1,
                          D[i - 1][j - 1] + cost)
    return D

In [9]:
D = med(source, target)
pd.DataFrame(D, index = ["#"] + list(source), columns = ["#"] + list(target))

,#,e,x,e,c,u,t,i,o,n
#,0,1,2,3,4,5,6,7,8,9
i,1,2,3,4,5,6,7,6,7,8
n,2,3,4,5,6,7,8,7,8,7
t,3,4,5,6,7,8,7,8,9,8
e,4,3,4,5,6,7,8,9,10,9
n,5,4,5,6,7,8,9,10,11,10
t,6,5,6,7,8,9,8,9,10,11
i,7,6,7,8,9,10,9,8,9,10
o,8,7,8,9,10,11,10,9,8,9
n,9,8,9,10,11,12,11,10,9,8


In [10]:
D[-1][-1], distance.edit_distance(source, target, substitution_cost = 2)

(8, 8)

## 2. Mã hóa và áp dụng thuật toán Minimum Edit Distance cho cấp bậc câu

1. Tách từ của s, t
2. Tạo bộ từ điển: D = S ∪ T
3. Mã hóa bộ từ điển D
4. Dựa trên bộ từ điển mã hóa để mã hóa s và t
5. Tính MED(s, t)

Dữ liệu lấy từ corpus Brown (thể loại news).

In [11]:
news = brown.sents(categories = "news")

source = " ".join(news[0])
target = " ".join(news[1])

print(source)
print(target)

The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .
The jury further said in term-end presentments that the City Executive Committee , which had over-all charge of the election , `` deserves the praise and thanks of the City of Atlanta '' for the manner in which the election was conducted .


In [12]:
source_tokens = source.lower().split()
target_tokens = target.lower().split()

In [13]:
vocab = sorted(set(source_tokens) | set(target_tokens))
len(vocab)

46

In [14]:
word2id = {word: idx for idx, word in enumerate(vocab)}

In [15]:
source_encoded = [word2id[w] for w in source_tokens]
target_encoded = [word2id[w] for w in target_tokens]

print(source_encoded)
print(target_encoded)

[42, 20, 13, 22, 27, 38, 19, 4, 25, 30, 8, 37, 35, 15, 36, 3, 29, 16, 0, 41, 6, 26, 43, 32, 2]
[42, 27, 21, 38, 24, 39, 34, 41, 42, 10, 17, 11, 1, 45, 23, 31, 9, 30, 42, 15, 1, 3, 14, 42, 33, 5, 40, 30, 42, 10, 30, 7, 0, 18, 42, 28, 24, 45, 42, 15, 44, 12, 2]


In [16]:
distance.edit_distance(source_encoded, target_encoded, substitution_cost = 1)

38

In [17]:
distance.edit_distance(source_encoded, target_encoded, substitution_cost = 2)

52

### Áp dụng trên nhiều câu của corpus: tìm câu gần nhất với câu đầu tiên

In [18]:
sample = [" ".join(s) for s in news[:500]]

tokens = [s.lower().split() for s in sample]
vocab = sorted(set(w for t in tokens for w in t))
word2id = {word: idx for idx, word in enumerate(vocab)}
encoded = [[word2id[w] for w in t] for t in tokens]

dist = [distance.edit_distance(encoded[0], e) for e in encoded[1:]]
nearest = dist.index(min(dist)) + 1

In [19]:
print(sample[0])
print(sample[nearest])
print("MED =", min(dist))

The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .
The jury said it did find that many of Georgia's registration and election laws `` are outmoded or inadequate and often ambiguous '' .
MED = 20


## 3. Backtrace for computing alignments

- Lần ngược từ ô cuối cùng về ô gốc để tìm alignment giữa hai chuỗi
- Mỗi ô lưu một **list** con trỏ (tối đa 3 phần tử): `LEFT` = insertion, `DOWN` = deletion, `DIAG` = substitution
- Time O(nm), Space O(nm), Backtrace O(n + m)

In [20]:
def med_backtrace(source, target, sub_cost = 2):
    D = med(source, target, sub_cost)
    ptr = [[[] for _ in range(len(target) + 1)] for _ in range(len(source) + 1)]
    for i in range(1, len(source) + 1):
        ptr[i][0] = ["DOWN"]
    for j in range(1, len(target) + 1):
        ptr[0][j] = ["LEFT"]
    for i in range(1, len(source) + 1):
        for j in range(1, len(target) + 1):
            cost = 0 if source[i - 1] == target[j - 1] else sub_cost
            if D[i - 1][j] + 1 == D[i][j]:
                ptr[i][j].append("DOWN")
            if D[i][j - 1] + 1 == D[i][j]:
                ptr[i][j].append("LEFT")
            if D[i - 1][j - 1] + cost == D[i][j]:
                ptr[i][j].append("DIAG")
    return D, ptr

In [21]:
D, ptr = med_backtrace(source = "intention", target = "execution")
ptr[5][5]

['DOWN', 'LEFT', 'DIAG']

In [22]:
def alignment(source, target, ptr):
    i, j = len(source), len(target)
    top, bottom = [], []
    while i > 0 or j > 0:
        move = ptr[i][j][-1]          # ưu tiên đi theo đường chéo (DIAG)
        if move == "DIAG":
            top.append(source[i - 1])
            bottom.append(target[j - 1])
            i, j = i - 1, j - 1
        elif move == "DOWN":
            top.append(source[i - 1])
            bottom.append("*")
            i -= 1
        else:
            top.append("*")
            bottom.append(target[j - 1])
            j -= 1
    return top[::-1], bottom[::-1]

In [23]:
top, bottom = alignment("intention", "execution", ptr)

print(" ".join(top))
print(" ".join(bottom))

i n t e * n t i o n
* e x e c u t i o n


## 4. Weighted Min Edit Distance

Bổ sung các bảng trọng số `del`, `ins`, `sub` để một số lỗi (ví dụ gõ nhầm phím gần nhau) tốn chi phí thấp hơn.

In [24]:
def weighted_med(source, target, del_w, ins_w, sub_w):
    D = [[0] * (len(target) + 1) for _ in range(len(source) + 1)]
    for i in range(1, len(source) + 1):
        D[i][0] = D[i - 1][0] + del_w.get(source[i - 1], 1)
    for j in range(1, len(target) + 1):
        D[0][j] = D[0][j - 1] + ins_w.get(target[j - 1], 1)
    for i in range(1, len(source) + 1):
        for j in range(1, len(target) + 1):
            if source[i - 1] == target[j - 1]:
                sub = 0
            else:
                sub = sub_w.get((source[i - 1], target[j - 1]), 2)
            D[i][j] = min(D[i - 1][j] + del_w.get(source[i - 1], 1),
                          D[i][j - 1] + ins_w.get(target[j - 1], 1),
                          D[i - 1][j - 1] + sub)
    return D

In [25]:
sub_w = {("e", "o"): 1, ("r", "c"): 1}     # các cặp dễ nhầm lẫn thì chi phí thấp hơn

for word in ["actress", "cress", "caress", "access", "across", "acres"]:
    print(word, weighted_med("acress", word, {}, {}, {})[-1][-1], weighted_med("acress", word, {}, {}, sub_w)[-1][-1])

actress 1 1
cress 1 1
caress 2 2
access 2 1
across 2 1
acres 1 1


## 5. Bài tập thêm: Needleman-Wunsch và Smith-Waterman

- Trong Computational Biology người ta tối đa hóa *similarity* (score) thay vì tối thiểu hóa distance nên dùng `max`
- Needleman-Wunsch (slide 40): global alignment
- Smith-Waterman (slide 46): local alignment, thêm 0 vào `max` để bỏ qua vùng khớp kém

In [26]:
def needleman_wunsch(x, y, match = 1, mismatch = -1, gap = 1):
    F = [[0] * (len(y) + 1) for _ in range(len(x) + 1)]
    for i in range(len(x) + 1):
        F[i][0] = -i * gap
    for j in range(len(y) + 1):
        F[0][j] = -j * gap
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            s = match if x[i - 1] == y[j - 1] else mismatch
            F[i][j] = max(F[i - 1][j] - gap,
                          F[i][j - 1] - gap,
                          F[i - 1][j - 1] + s)
    return F

In [27]:
def smith_waterman(x, y, match = 1, mismatch = -1, gap = 1):
    F = [[0] * (len(y) + 1) for _ in range(len(x) + 1)]
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            s = match if x[i - 1] == y[j - 1] else mismatch
            F[i][j] = max(0,
                          F[i - 1][j] - gap,
                          F[i][j - 1] - gap,
                          F[i - 1][j - 1] + s)
    return F

In [28]:
x, y = "ATCAT", "ATTATC"

F = needleman_wunsch(x, y)
pd.DataFrame(F, index = ["#"] + list(x), columns = ["#"] + list(y))

,#,A,T,T,A,T,C
#,0,-1,-2,-3,-4,-5,-6
A,-1,1,0,-1,-2,-3,-4
T,-2,0,2,1,0,-1,-2
C,-3,-1,1,1,0,-1,0
A,-4,-2,0,0,2,1,0
T,-5,-3,-1,1,1,3,2


In [29]:
F = smith_waterman(x, y)
pd.DataFrame(F, index = ["#"] + list(x), columns = ["#"] + list(y))

,#,A,T,T,A,T,C
#,0,0,0,0,0,0,0
A,0,1,0,0,1,0,0
T,0,0,2,1,0,2,1
C,0,0,1,1,0,1,3
A,0,1,0,0,2,1,2
T,0,0,2,1,1,3,2


In [30]:
print("Needleman-Wunsch (global):", needleman_wunsch(x, y)[-1][-1])
print("Smith-Waterman (local)    :", max(max(row) for row in F))

Needleman-Wunsch (global): 2
Smith-Waterman (local)    : 3


# 2. MÔ HÌNH NGÔN NGỮ 2: THUẬT TOÁN N-GRAMS

- Dùng thống kê và xác suất để biết từ nào xuất hiện nhiều hơn trong câu/văn bản
- **Chain Rule**: P(x1,...,xn) = P(x1)P(x2|x1)P(x3|x1,x2)...P(xn|x1,...,xn-1)
- **Markov Assumption**: chỉ xét N-1 từ liền trước thay vì toàn bộ lịch sử
- Có thể dùng trigrams, 4-grams, 5-grams; không nên dùng ràng buộc phụ thuộc quá xa

## Dữ liệu: corpus Brown (thể loại news)

In [31]:
sentences = [[w.lower() for w in s] for s in brown.sents(categories = "news")]

split = int(len(sentences) * 0.8)
train, test = sentences[:split], sentences[split:]

len(train), len(test)

(3698, 925)

## 1. Mô hình con 1: Unigram

In [32]:
train_words = [w for s in train for w in s]

unigram = nltk.FreqDist(train_words)
unigram.most_common(10)

[('the', 5072),
 (',', 4029),
 ('.', 3211),
 ('of', 2321),
 ('and', 1740),
 ('a', 1710),
 ('to', 1649),
 ('in', 1592),
 ('for', 787),
 ('that', 644)]

In [33]:
unigram.freq("the")

0.06356287987969171

## 2. Mô hình con 2: Bigram

In [34]:
bigrams = [b for s in train for b in nltk.bigrams(["<s>"] + s + ["</s>"])]

bigram = nltk.ConditionalFreqDist(bigrams)
bigram["of"].most_common(5)

[('the', 695), ('a', 91), ('his', 31), ('their', 23), ('this', 20)]

In [35]:
bigram["of"].freq("the")      # P(the | of)

0.2994398965962947

## 3. Tính giá trị N-grams: xác suất của câu (trong log space)

In [36]:
def sentence_logprob(sentence):
    logp = 0
    for prev, w in nltk.bigrams(["<s>"] + sentence + ["</s>"]):
        p = bigram[prev].freq(w)
        if p == 0:
            return float("-inf")
        logp += math.log(p)
    return logp

In [37]:
print(train[0])
print(sentence_logprob(train[0]))

['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', "atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']
-78.92709163729535


In [38]:
print(test[0])
print(sentence_logprob(test[0]))

['once', 'more', 'the', 'fallacious', 'equation', 'is', 'advanced', 'to', 'argue', 'that', 'since', 'business', 'is', 'restricted', 'under', 'the', 'anti-monopoly', 'laws', ',', 'there', 'must', 'be', 'a', 'corresponding', 'restriction', 'against', 'labor', 'unions', ':', 'the', 'law', 'must', 'treat', 'everybody', 'equally', '.']
-inf


### Shannon Visualization Method

In [39]:
random.seed(1)

word, output = "<s>", []
while word != "</s>" and len(output) < 30:
    word = random.choices(list(bigram[word]), weights = list(bigram[word].values()))[0]
    output.append(word)

" ".join(output)

'the northeast and thomas glennon , treasurer . </s>'

## 4. Đánh giá mô hình N-grams

- Chia **training** và **test** (tập test là dữ liệu chưa từng thấy)
- **Perplexity** càng thấp thì mô hình càng tốt
- Từ chưa từng thấy được thay bằng `<UNK>`

In [40]:
lexicon = {w for w, c in unigram.items() if c >= 2}

train_unk = [[w if w in lexicon else "<UNK>" for w in s] for s in train]
test_unk = [[w if w in lexicon else "<UNK>" for w in s] for s in test]

len(lexicon)

5363

In [41]:
from nltk.lm import MLE, Laplace, StupidBackoff, WittenBellInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends

test_bigrams = [b for s in test_unk for b in nltk.bigrams(pad_both_ends(s, n = 2))]

In [42]:
def perplexity_of(lm):
    train_data, vocab = padded_everygram_pipeline(2, train_unk)
    lm.fit(train_data, vocab)
    return lm.perplexity(test_bigrams)

In [43]:
perplexity_of(MLE(2))

inf

## 5. Kỹ thuật Laplace Smoothing cho vấn đề xác suất bằng 0

- Bigram chưa xuất hiện trong training sẽ có xác suất 0, không tính được perplexity
- Add-one: cộng 1 vào mọi bộ đếm: P(w|prev) = (C(prev,w) + 1) / (C(prev) + V)

In [44]:
V = len(unigram)

for prev, w in [("of", "the"), ("the", "of"), ("of", "said")]:
    mle = bigram[prev].freq(w)
    laplace = (bigram[prev][w] + 1) / (bigram[prev].N() + V)
    print(f"P({w}|{prev})  MLE = {mle:.6f}   Laplace = {laplace:.6f}")

P(the|of)  MLE = 0.299440   Laplace = 0.050640
P(of|the)  MLE = 0.000000   Laplace = 0.000061
P(said|of)  MLE = 0.000000   Laplace = 0.000073


In [45]:
perplexity_of(Laplace(2))

706.1872243625726

## 6. Interpolation, Backoff và Web-scale LMs

- **Interpolation**: kết hợp xác suất các bậc bằng tổng có trọng số, bigram bằng 0 thì unigram vẫn giúp xác suất khác 0
- **Backoff**: dùng bigram nếu có bằng chứng, nếu không lùi xuống unigram
- **Web-scale**: pruning (bỏ N-grams có count nhỏ), *stupid backoff*

In [46]:
lam = 0.7

for prev, w in [("of", "the"), ("the", "of"), ("of", "said")]:
    p = lam * bigram[prev].freq(w) + (1 - lam) * unigram.freq(w)
    print(f"P({w}|{prev}) = {p:.6f}")

P(the|of) = 0.228677
P(of|the) = 0.008726
P(said|of) = 0.001402


In [47]:
perplexity_of(WittenBellInterpolated(2))

166.09614405485092

In [48]:
perplexity_of(StupidBackoff(alpha = 0.4, order = 2))

144.35024098233475

In [49]:
bigram_counts = nltk.FreqDist(bigrams)
pruned = {b: c for b, c in bigram_counts.items() if c > 1}

len(bigram_counts), len(pruned)

(49988, 8768)

# 3. SPELLING CORRECTION AND THE NOISY CHANNEL

- **Non-word error**: từ không có trong bộ từ điển; **real-word error**: từ có nghĩa nhưng sai ngữ cảnh
- Sửa lỗi: sinh các ứng viên gần với từ lỗi (Edit Distance), rồi chọn ứng viên có P(x|w)·P(w) lớn nhất
    - P(x|w): channel model, P(w): language model (N-grams)

## 1. Phát hiện lỗi non-word bằng bộ từ điển

In [50]:
brown_words = [w.lower() for w in brown.words() if w.isalpha()]

dictionary = set(w.lower() for w in words.words()) | set(brown_words)
len(dictionary)

251972

In [51]:
text = "the office is about fifteen minutes from my huose".split()

[w for w in text if w not in dictionary]

['huose']

## 2. Sinh ứng viên: các từ nằm trong khoảng Damerau-Levenshtein bằng 1

In [52]:
def edits1(word):
    letters = "abcdefghijklmnopqrstuvwxyz"
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [a + b[1:] for a, b in splits if b]
    transposes = [a + b[1] + b[0] + b[2:] for a, b in splits if len(b) > 1]
    replaces = [a + c + b[1:] for a, b in splits if b for c in letters]
    inserts = [a + c + b for a, b in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)

In [53]:
candidates = edits1("huose") & dictionary
candidates

{'hoose', 'hose', 'house', 'huse'}

In [54]:
for w in candidates:
    print(w, distance.edit_distance("huose", w, transpositions = True))

huse 1
house 1
hoose 1
hose 1


## 3. Noisy Channel: chọn ứng viên có P(x|w)·P(w) lớn nhất

- P(w) lấy từ tần suất trong corpus Brown
- P(x|w) lấy từ confusion matrix của slide (Kernighan, Church, Gale 1990)

In [55]:
brown_freq = nltk.FreqDist(brown_words)

channel = {"actress": 0.000117, "cress": 0.00000144, "caress": 0.00000164,
           "access": 0.000000209, "across": 0.0000093, "acres": 0.0000321}

for w, p_x_w in sorted(channel.items(), key = lambda r: -r[1] * brown_freq.freq(r[0])):
    print(f"{w:<8} P(x|w) = {p_x_w:.9f}   P(w) = {brown_freq.freq(w):.8f}   1e9*P(x|w)P(w) = {1e9 * p_x_w * brown_freq.freq(w):.3f}")

across   P(x|w) = 0.000009300   P(w) = 0.00028725   1e9*P(x|w)P(w) = 2.671
acres    P(x|w) = 0.000032100   P(w) = 0.00004482   1e9*P(x|w)P(w) = 1.439
actress  P(x|w) = 0.000117000   P(w) = 0.00000611   1e9*P(x|w)P(w) = 0.715
access   P(x|w) = 0.000000209   P(w) = 0.00002445   1e9*P(x|w)P(w) = 0.005
caress   P(x|w) = 0.000001640   P(w) = 0.00000102   1e9*P(x|w)P(w) = 0.002
cress    P(x|w) = 0.000001440   P(w) = 0.00000000   1e9*P(x|w)P(w) = 0.000


### Dùng bigram language model để lấy ngữ cảnh: `my ___`

In [56]:
brown_bigram = nltk.ConditionalFreqDist(nltk.bigrams(brown_words))
V = len(brown_freq)

for w in candidates:
    p = (brown_bigram["my"][w] + 1) / (brown_bigram["my"].N() + V)      # P(w | my), add-one
    print(f"{w:<8} P({w}|my) = {p:.8f}")

huse     P(huse|my) = 0.00002407
house    P(house|my) = 0.00012033
hoose    P(hoose|my) = 0.00002407
hose     P(hose|my) = 0.00002407


## 4. Real-word spelling error

Với mỗi từ cần xét thêm xác suất không lỗi P(w|w): 0.90 (1 lỗi/10 từ), 0.95 (1/20), 0.99 (1/100), 0.995 (1/200).

Ví dụ `thew` (Peter Norvig): x = thew, các ứng viên w gồm cả chính nó.

In [57]:
thew = {"the": 0.000007, "thew": 0.95, "thaw": 0.001, "threw": 0.000008, "thwe": 0.000003}

for w, p_x_w in sorted(thew.items(), key = lambda r: -r[1] * brown_freq.freq(r[0])):
    print(f"{w:<6} 1e9*P(x|w)P(w) = {1e9 * p_x_w * brown_freq.freq(w):.3f}")

the    1e9*P(x|w)P(w) = 498.919
thaw   1e9*P(x|w)P(w) = 6.112
threw  1e9*P(x|w)P(w) = 0.375
thew   1e9*P(x|w)P(w) = 0.000
thwe   1e9*P(x|w)P(w) = 0.000
